# IEEE-CIS Fraud Detection: Neo4j Graph Features + Vertex AI

**Key idea**: Graph features extracted offline from Neo4j (entity fraud rates, FastRP node embeddings) significantly improve fraud detection versus tabular ML alone.

**This notebook**:
1. Shows the Neo4j graph data model
2. Loads pre-computed graph features and node embeddings (no live database connection needed)
3. Trains tabular baseline vs graph-enhanced LightGBM models
4. Reports the lift from graph context

---

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set USE_GCS = True and fill in GCS_BUCKET when running on Vertex AI Workbench
USE_GCS    = False
GCS_BUCKET = 'your-gcs-bucket'
GCS_PREFIX = 'neo4j-fraud-detection'

import os

def artifact(filename):
    if USE_GCS:
        return f'gs://{GCS_BUCKET}/{GCS_PREFIX}/artifacts/{filename}'
    return os.path.join('..', 'artifacts', filename)

def data_file(filename):
    if USE_GCS:
        return f'gs://{GCS_BUCKET}/{GCS_PREFIX}/{filename}'
    return os.path.join('..', filename)

# ── Imports ───────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from IPython.display import display, Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)
import lightgbm as lgb

print('Setup complete.')

---
## 1. The Neo4j Graph Data Model

The fraud network is modelled as a **bipartite graph**: `Transaction` nodes at the center connected to shared entity nodes (Card, EmailDomain, BillingAddress, Device, OSBrowser, ProxyType). Fraud rings become visible as dense clusters where many fraud-labeled transactions share the same card, device, or address.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
img = mpimg.imread(os.path.join('..', 'docs', 'Fraud_Datamodel_Graph_DB.png'))
ax.imshow(img)
ax.axis('off')
ax.set_title('Neo4j Graph Data Model - IEEE-CIS Fraud Detection', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

---
## 2. Load Pre-computed Graph Artifacts

The following artifacts were generated **offline** using Neo4j Aura Pro + GDS and stored as Parquet files. No live database connection is needed for this demo.

| Artifact | Description | Size |
|---|---|---|
| `graph_features_v2.parquet` | 22 entity-level fraud rate features per transaction | 11 MB |
| `transaction_embeddings_v2.parquet` | 64-dim FastRP node embeddings | 180 MB |

In [ ]:
# Graph features (entity fraud rates, proxy flags, temporal chain signal)
print('Loading graph features ...')
graph_feat = pd.read_parquet(artifact('graph_features_v2.parquet'))
graph_feat = graph_feat.rename(columns={'transactionId': 'TransactionID'})
print(f'  graph_features_v2 : {graph_feat.shape}  columns: {list(graph_feat.columns[:6])} ...')

# FastRP node embeddings (64 dims)
print('Loading FastRP embeddings ...')
embeddings = pd.read_parquet(artifact('transaction_embeddings_v2.parquet'))
embeddings = embeddings.rename(columns={'transactionId': 'TransactionID'})
print(f'  embeddings_v2     : {embeddings.shape}')

# Raw transaction data (labels + tabular features)
print('Loading transaction CSV ...')
transactions = pd.read_csv(data_file('train_transaction.csv'))
print(f'  train_transaction : {transactions.shape}')

# Merge everything on TransactionID
df = transactions.merge(graph_feat, on='TransactionID', how='left')
df = df.merge(embeddings, on='TransactionID', how='left')
print(f'\nMerged dataset : {df.shape}')
print(f'Fraud rate     : {df["isFraud"].mean():.3%}')

In [ ]:
# Temporal train/val split - same boundary used when computing graph features
# (avoids leakage: entity fraud rates are computed only on training transactions)
SPLIT_DT = 12_528_000
train = df[df['TransactionDT'] <= SPLIT_DT].copy()
val   = df[df['TransactionDT'] >  SPLIT_DT].copy()

print(f'Train : {len(train):>7,} rows  |  fraud rate: {train["isFraud"].mean():.3%}')
print(f'Val   : {len(val):>7,} rows  |  fraud rate: {val["isFraud"].mean():.3%}')

---
## 3. Graph Feature Signal Strength

Each bar chart shows how strongly a graph-derived feature separates fraud from legitimate transactions. The ratio (fraud mean / legit mean) measures discriminative power.

In [ ]:
GRAPH_RATE_COLS = [
    'card_fraud_rate', 'device_fraud_rate', 'billing_fraud_rate',
    'os_browser_fraud_rate', 'recip_email_fraud_rate', 'prev_card_is_fraud'
]

palette = {1: '#e74c3c', 0: '#3498db'}
labels  = {1: 'Fraud', 0: 'Legitimate'}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Graph Feature Distributions: Fraud vs Legitimate Transactions', fontsize=14)

for ax, col in zip(axes.flat, GRAPH_RATE_COLS):
    for lbl in [0, 1]:
        data = train.loc[train['isFraud'] == lbl, col].dropna()
        ax.hist(data, bins=30, alpha=0.65, density=True,
                color=palette[lbl], label=labels[lbl])
    fraud_mean = train.loc[train['isFraud'] == 1, col].mean()
    legit_mean = train.loc[train['isFraud'] == 0, col].mean()
    ratio = fraud_mean / legit_mean if legit_mean > 1e-9 else float('inf')
    ax.set_title(f'{col}\nfraud/legit ratio: {ratio:.1f}x', fontsize=10)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. FastRP Node Embedding Visualization

Neo4j GDS computed 64-dimensional **FastRP** embeddings for every Transaction node. Each embedding encodes a transaction's position in the fraud network (its neighborhood of cards, devices, email domains, and temporal card chain).

We project from 64D to 2D using PCA to visualize the separation between fraud and legitimate transactions.

In [ ]:
EMB_COLS = [f'emb_{i}' for i in range(64)]

# Sample 10K transactions for visualization
sample = train.sample(n=min(10_000, len(train)), random_state=42)
X_sample = sample[EMB_COLS].fillna(0).values
y_sample = sample['isFraud'].values

pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_sample)

fig, ax = plt.subplots(figsize=(12, 7))
for lbl in [0, 1]:
    mask = y_sample == lbl
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=palette[lbl], alpha=0.25 if lbl == 0 else 0.7,
        s=5 if lbl == 0 else 9,
        label=f'{labels[lbl]} ({mask.sum():,})'
    )

ax.set_title(
    f'FastRP Node Embeddings - PCA Projection (2D from 64 dims)\n'
    f'PC1: {pca.explained_variance_ratio_[0]:.1%} variance | '
    f'PC2: {pca.explained_variance_ratio_[1]:.1%} variance',
    fontsize=12
)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(fontsize=12, markerscale=3)
plt.tight_layout()
plt.show()

---
## 5. Feature Engineering - Prepare Feature Sets

In [ ]:
# Encode categorical columns (label encoding for LightGBM)
CAT_COLS = ['card4', 'card6', 'ProductCD', 'P_emaildomain', 'R_emaildomain']
for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str).fillna('nan'))

# Re-split after encoding
train = df[df['TransactionDT'] <= SPLIT_DT].copy()
val   = df[df['TransactionDT'] >  SPLIT_DT].copy()

# Feature column sets
EXCLUDE  = {'TransactionID', 'TransactionDT', 'isFraud', 'transactionId'}
GFT_COLS = [c for c in graph_feat.columns
            if c != 'TransactionID' and not c.startswith('component')]
TAB_COLS = [c for c in df.columns
            if c not in EXCLUDE and c not in GFT_COLS and c not in EMB_COLS]
ALL_COLS = TAB_COLS + GFT_COLS + EMB_COLS

print(f'Tabular features  : {len(TAB_COLS)}')
print(f'Graph features    : {len(GFT_COLS)}')
print(f'Embedding dims    : {len(EMB_COLS)}')
print(f'Total (enhanced)  : {len(ALL_COLS)}')

SCALE_POS = (train['isFraud'] == 0).sum() / (train['isFraud'] == 1).sum()
print(f'\nClass imbalance weight (scale_pos_weight): {SCALE_POS:.1f}')

---
## 6. Experiment 1 - Tabular Baseline

LightGBM trained on raw tabular features only. Each transaction is treated as an independent row - no graph context.

In [ ]:
print('Training tabular baseline ...')

X_tr  = train[TAB_COLS];  y_tr  = train['isFraud']
X_val = val[TAB_COLS];    y_val = val['isFraud']

clf_base = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    scale_pos_weight=SCALE_POS, random_state=42, n_jobs=-1, verbose=-1
)
clf_base.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

prob_base = clf_base.predict_proba(X_val)[:, 1]
roc_base  = roc_auc_score(y_val, prob_base)
pr_base   = average_precision_score(y_val, prob_base)

print(f'\nBaseline  ROC-AUC : {roc_base:.4f}')
print(f'Baseline  PR-AUC  : {pr_base:.4f}  (primary metric)')

---
## 7. Experiment 2 - Graph-Enhanced Model

Same LightGBM architecture, but features now include:
- **22 graph features**: entity fraud rates (card, device, billing address, email domain, OS/browser, proxy flag, temporal card chain)
- **64 FastRP embedding dimensions**: each transaction's structural position in the fraud network

In [ ]:
print('Training graph-enhanced model ...')

X_tr_g  = train[ALL_COLS].fillna(-1);  y_tr_g  = train['isFraud']
X_val_g = val[ALL_COLS].fillna(-1);    y_val_g = val['isFraud']

clf_graph = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    scale_pos_weight=SCALE_POS, random_state=42, n_jobs=-1, verbose=-1
)
clf_graph.fit(
    X_tr_g, y_tr_g,
    eval_set=[(X_val_g, y_val_g)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

prob_graph = clf_graph.predict_proba(X_val_g)[:, 1]
roc_graph  = roc_auc_score(y_val_g, prob_graph)
pr_graph   = average_precision_score(y_val_g, prob_graph)

print(f'\nGraph-Enhanced  ROC-AUC : {roc_graph:.4f}')
print(f'Graph-Enhanced  PR-AUC  : {pr_graph:.4f}  (primary metric)')

---
## 8. Results Comparison

In [ ]:
def best_threshold(y_true, prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        f = f1_score(y_true, prob >= t, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t

t_base  = best_threshold(y_val,   prob_base)
t_graph = best_threshold(y_val_g, prob_graph)
p_base  = (prob_base  >= t_base).astype(int)
p_graph = (prob_graph >= t_graph).astype(int)

metric_names  = ['ROC-AUC', 'PR-AUC', 'F1 (fraud)', 'Precision', 'Recall']
base_scores   = [
    roc_base, pr_base,
    f1_score(y_val,   p_base),
    precision_score(y_val,   p_base, zero_division=0),
    recall_score(y_val,   p_base)
]
graph_scores  = [
    roc_graph, pr_graph,
    f1_score(y_val_g, p_graph),
    precision_score(y_val_g, p_graph, zero_division=0),
    recall_score(y_val_g, p_graph)
]

results = pd.DataFrame({
    'Metric':           metric_names,
    'Tabular Baseline': base_scores,
    'Graph-Enhanced':   graph_scores,
})
results['Delta']       = results['Graph-Enhanced'] - results['Tabular Baseline']
results['Improvement'] = (results['Delta'] / results['Tabular Baseline'] * 100).map('{:+.1f}%'.format)
print(results.round(4).to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison: Tabular Baseline vs Graph-Enhanced', fontsize=14)

x = np.arange(len(metric_names))
w = 0.35
b1 = ax1.bar(x - w/2, base_scores,  w, label='Tabular Baseline', color='#3498db')
b2 = ax1.bar(x + w/2, graph_scores, w, label='Graph-Enhanced',   color='#e74c3c')
ax1.set_xticks(x)
ax1.set_xticklabels(metric_names, rotation=15, ha='right')
ax1.set_ylim(0.5, 1.05)
ax1.legend(fontsize=10)
ax1.set_title('All Metrics')
ax1.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
ax1.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)

ax2.bar(['Tabular\nBaseline', 'Graph-\nEnhanced'],
        [pr_base, pr_graph],
        color=['#3498db', '#e74c3c'], width=0.4)
ax2.set_ylim(0, 1.0)
ax2.set_title('PR-AUC - Primary Metric for Fraud Detection')
for i, v in enumerate([pr_base, pr_graph]):
    ax2.text(i, v + 0.012, f'{v:.4f}', ha='center', fontsize=13, fontweight='bold')
pct = (pr_graph - pr_base) / pr_base * 100
ax2.set_xlabel(f'Graph model improvement: {pct:+.1f}% PR-AUC', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
feat_imp = pd.DataFrame({
    'feature':    clf_graph.feature_name_,
    'importance': clf_graph.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

def feat_type(name):
    if name in GFT_COLS: return 'Graph'
    if name in EMB_COLS: return 'Embedding'
    return 'Tabular'

feat_imp['type'] = feat_imp['feature'].map(feat_type)
top25 = feat_imp.head(25)

color_map = {'Graph': '#e74c3c', 'Embedding': '#f39c12', 'Tabular': '#3498db'}
colors = top25['type'].map(color_map).tolist()

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top25['feature'], top25['importance'], color=colors)
ax.invert_yaxis()
ax.set_title('Top 25 Feature Importances - Graph-Enhanced Model', fontsize=13)
ax.set_xlabel('Importance')

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=t) for t, c in color_map.items()]
ax.legend(handles=legend_handles, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()